# 02 · Limpieza y calidad en Silver

Ejecutar en el nodo primario de EMR con el kernel **American Tower - PySpark**, en un kernel nuevo y en orden. Cada celda muestra la evidencia que produce. Los registros de Spark y el resultado quedan enlazados por el identificador de corrida.

El manifiesto Bronze fija los originales de la ejecución. Silver asigna las teselas a polígonos distritales, documenta
rechazos y publica dimensiones, cobertura OSIPTEL y población de WorldPop. Las tablas se escriben en el bucket Silver
bajo `run_id=…/<tabla>/`.

## 1. Configuración y manifiesto de entrada

In [3]:
import pandas as pd
from IPython.display import display


def show_values(values):
    show_rows(values.items(), ["Indicador", "Valor"])


def show_rows(rows, columns=None):
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(rows, columns=columns))

REGION = "us-east-1"
OPERATIONS_BUCKET = "american-tower-dev-operations-1dd72256126674a5893dfca3df"
SILVER_BUCKET = "american-tower-dev-silver-2ad71fded155d2bd52df8f836e"
BRONZE_RUN_URI = "s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T071603Z-190d23d89e/bronze/_SUCCESS.json"  # Pegar la manifest_uri de entrada.
EXECUTORS = 10
EXECUTOR_CORES = 4
EXECUTOR_MEMORY = "8g"
EXECUTOR_OVERHEAD = "3g"
SHUFFLE_PARTITIONS = EXECUTORS * 12

TEMP_DIR = "/mnt"

show_values({"Región": REGION, "Manifiesto de entrada": BRONZE_RUN_URI or "Pendiente de configurar", "Bucket Silver": SILVER_BUCKET, "Bucket Operations": OPERATIONS_BUCKET, "Executors solicitados": EXECUTORS})

,Indicador,Valor
0,Región,us-east-1
1,Manifiesto de entrada,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T071603Z-190d23d89e/bronze/_SUCCESS.json
2,Bucket Silver,american-tower-dev-silver-2ad71fded155d2bd52df8f836e
3,Bucket Operations,american-tower-dev-operations-1dd72256126674a5893dfca3df
4,Executors solicitados,10


## 2. Inventario Bronze recibido

In [4]:
import json
import uuid
from datetime import datetime, timezone

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

s3 = boto3.client("s3", region_name=REGION, config=Config(retries={"mode": "standard", "max_attempts": 5}))
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:10]


def s3_path(uri):
    if not isinstance(uri, str) or not uri.startswith("s3://"):
        raise ValueError(f"Se requiere una URI S3 completa: {uri!r}")
    bucket, slash, key = uri.removeprefix("s3://").partition("/")
    if not bucket or not slash or not key:
        raise ValueError(f"Se requiere una URI S3 completa: {uri}")
    return bucket, key


def missing_object(error):
    return error.response["Error"]["Code"] in {"NoSuchKey", "NotFound", "404"}


def read_json(uri):
    bucket, key = s3_path(uri)
    try:
        response = s3.get_object(Bucket=bucket, Key=key)
    except ClientError as error:
        if missing_object(error):
            raise FileNotFoundError(f"No existe el manifiesto {uri}") from error
        raise
    with response["Body"] as stream:
        return json.load(stream)


def save_json(uri, value):
    bucket, key = s3_path(uri)
    s3.put_object(
        Bucket=bucket, Key=key, IfNoneMatch="*", ContentType="application/json",
        Body=json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False).encode("utf-8"),
    )
    return uri



def read_manifest(uri, schema, parameter):
    if not uri:
        raise ValueError(f"Completar {parameter} con la manifest_uri que devolvió la etapa anterior.")
    document = read_json(uri)
    if document.get("schema") != schema or document.get("complete") is not True or not document.get("run_id"):
        raise ValueError(f"Manifiesto incompleto o de un tipo diferente a {schema}: {uri}")
    return document

from pathlib import PurePosixPath
import re

PERIOD_FILE = re.compile(r"^(\d{4})-(01|04|07|10)-01_performance_mobile_tiles\.parquet$")


def object_period(item):
    match = PERIOD_FILE.fullmatch(PurePosixPath(item["key"]).name)
    if not match:
        raise ValueError(f"Nombre de Parquet trimestral inesperado: {item['key']}")
    year, month = map(int, match.groups())
    return year * 4 + (month - 1) // 3


def validate_inventory(items):
    mobile = [item for item in items if item["source"] == "ookla" and item["key"].endswith(".parquet")]
    periods = [object_period(item) for item in mobile]
    if not periods or len(periods) != len(set(periods)):
        raise ValueError("Se requiere un solo Parquet Ookla móvil por trimestre.")
    coverage = [item for item in items if item["source"] == "osiptel_cobertura"
                and item["key"].lower().endswith(".csv")]
    districts = [item for item in items if item["source"] == "inei" and
                 re.fullmatch(r"distritos?\.(rar|zip|gpkg)", PurePosixPath(item["key"]).name.lower())]
    worldpop = [item for item in items if item["source"] == "worldpop"
                and PurePosixPath(item["key"]).name.lower() == "per_ppp_2020.tif"]
    if not all(len(group) == 1 for group in (coverage, districts, worldpop)):
        raise ValueError("El inventario requiere un CSV OSIPTEL, la capa distrital INEI y WorldPop 2020.")
    return mobile, coverage[0], districts[0], worldpop[0]


def verified_uri(item):
    try:
        header = s3.head_object(Bucket=item["bucket"], Key=item["key"])
    except ClientError as error:
        if missing_object(error):
            raise FileNotFoundError(f"No existe el objeto publicado en Bronze: {item['key']}") from error
        raise
    if (item.get("status") not in {"verified", "reused_verified"}
            or header["ContentLength"] != item["bytes"]
            or header.get("VersionId") != item.get("version_id")
            or header["Metadata"].get("sha256") != item["sha256"]):
        raise ValueError(f"El objeto difiere de su recibo Bronze: {item['key']}")
    return f"s3://{item['bucket']}/{item['key']}"
bronze = read_manifest(BRONZE_RUN_URI, "bronze-autonomo-v1", "BRONZE_RUN_URI")
mobile_items, coverage_item, district_item, worldpop_item = validate_inventory(bronze["objects"])
display(pd.DataFrame(bronze["objects"]).groupby("source", as_index=False).agg(
    objetos=("key", "count"), bytes=("bytes", "sum")
))
show_values({"Corrida Bronze": bronze["run_id"], "Trimestres de Ookla": len(mobile_items),
             "Archivo cartográfico": PurePosixPath(district_item["key"]).name,
             "Raster": PurePosixPath(worldpop_item["key"]).name})

/opt/american-tower/venv/lib64/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,source,objetos,bytes
0,inei,3,21407900
1,ookla,29,5676137401
2,osiptel_cobertura,1,7590550
3,osiptel_estaciones,1,740518
4,worldpop,1,704980975


,Indicador,Valor
0,Corrida Bronze,20261125T071603Z-190d23d89e
1,Trimestres de Ookla,29
2,Archivo cartográfico,Distrito.rar
3,Raster,per_ppp_2020.tif


## 3. Inicialización de Spark en YARN

In [5]:
import os
import sys
from contextlib import contextmanager
from pathlib import Path

SPARK_HOME = Path("/usr/lib/spark")
py4j = sorted((SPARK_HOME / "python/lib").glob("py4j-*.zip"))
if not py4j:
    raise RuntimeError("Abrir esta notebook en el nodo primario de Amazon EMR con Spark instalado.")
os.environ["SPARK_HOME"] = str(SPARK_HOME)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
sys.path[:0] = [str(SPARK_HOME / "python"), str(py4j[0])]

from pyspark import SparkContext, StorageLevel
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StructType


@contextmanager
def spark_session(stage):
    if SparkContext._active_spark_context is not None:
        raise RuntimeError("Reiniciar el kernel para que YARN aplique los parámetros de esta corrida.")
    # EMRFS necesita un prefijo existente para el registro de eventos.
    log_dir = f"run_id={RUN_ID}/eventos_spark/"
    s3.put_object(Bucket=OPERATIONS_BUCKET, Key=log_dir, Body=b"")
    spark = (
        SparkSession.builder.master("yarn").appName(f"american-tower-{stage}")
        .config("spark.submit.deployMode", "client")
        .config("spark.pyspark.python", sys.executable)
        .config("spark.executorEnv.PYSPARK_PYTHON", sys.executable)
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.executor.instances", str(EXECUTORS))
        .config("spark.executor.cores", str(EXECUTOR_CORES))
        .config("spark.executor.memory", EXECUTOR_MEMORY)
        .config("spark.executor.memoryOverhead", EXECUTOR_OVERHEAD)
        .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.eventLog.enabled", "true")
        .config("spark.eventLog.dir", f"s3://{OPERATIONS_BUCKET}/{log_dir}")
        .getOrCreate()
    )
    try:
        spark.sparkContext.setLogLevel("WARN")
        print(f"Aplicación Spark {spark.sparkContext.applicationId}", flush=True)
        yield spark
    finally:
        spark.stop()


def write_table(frame, uri, partitions=()):
    writer = frame.write.mode("errorifexists").option("compression", "snappy")
    if partitions:
        writer = writer.partitionBy(*partitions)
    writer.parquet(uri)
    return uri

from contextlib import ExitStack
from IPython import get_ipython
from urllib.parse import quote
from urllib.request import urlopen


def executor_usage(sc):
    if not sc.uiWebUrl:
        return {"estado": "interfaz Spark deshabilitada", "activos": None, "tareas": {}, "hosts": {}}
    app = quote(sc.applicationId, safe="")
    try:
        with urlopen(f"{sc.uiWebUrl.rstrip('/')}/api/v1/applications/{app}/allexecutors", timeout=4) as response:
            rows = [row for row in json.load(response) if row["id"] != "driver"]
    except (OSError, ValueError, KeyError) as error:
        return {"estado": f"interfaz no disponible ({type(error).__name__})",
                "activos": None, "tareas": {}, "hosts": {}}
    return {"estado": "lectura disponible", "activos": sum(row.get("isActive", False) for row in rows),
            "tareas": {row["id"]: row.get("totalTasks", 0) for row in rows},
            "hosts": {row["id"]: row.get("hostPort", "").rsplit(":", 1)[0] for row in rows}}


def show_spark(spark, events_uri):
    sc = spark.sparkContext
    show_values({"Aplicación": sc.applicationId, "Spark": spark.version, "Master": sc.master,
                 "Paralelismo base": sc.defaultParallelism, "Executors solicitados": EXECUTORS,
                 "vCores por executor": EXECUTOR_CORES,
                 "Heap y overhead por executor": f"{EXECUTOR_MEMORY} + {EXECUTOR_OVERHEAD}",
                 "Particiones shuffle": spark.conf.get("spark.sql.shuffle.partitions"),
                 "Eventos Spark": events_uri})


def open_stage(stage):
    events = get_ipython().events
    resources = ExitStack()

    def stop_on_failure(result):
        if result.error_before_exec or result.error_in_exec:
            resources.close()

    events.register("post_run_cell", stop_on_failure)
    resources.callback(events.unregister, "post_run_cell", stop_on_failure)
    try:
        spark = resources.enter_context(spark_session(stage))
    except BaseException:
        resources.close()
        raise
    events_uri = f"s3://{OPERATIONS_BUCKET}/run_id={RUN_ID}/eventos_spark/"
    return spark, resources, events_uri

spark, resources, events_uri = open_stage("silver")
show_spark(spark, events_uri)

Nov 25, 2026 7:25:49 AM org.apache.spark.launcher.Log4jHotPatchOption staticJavaAgentOption

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 07:25:52 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Aplicación Spark application_1795587001536_0004


,Indicador,Valor
0,Aplicación,application_1795587001536_0004
1,Spark,3.5.0-amzn-1
2,Master,yarn
3,Paralelismo base,2
4,Executors solicitados,10
5,vCores por executor,4
6,Heap y overhead por executor,8g + 3g
7,Particiones shuffle,120
8,Eventos Spark,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T072542Z-e92a810559/eventos_spark/


## 4. Preparación cartográfica y revisión del raster

In [6]:
import hashlib
import math
import subprocess
from tempfile import TemporaryDirectory

import geopandas as gpd
import numpy as np
import rasterio
import shapely
from rasterio.mask import mask


def downloaded_object(item, folder):
    uri = verified_uri(item)
    bucket, key = s3_path(uri)
    path = folder / PurePosixPath(key).name
    extra = {"VersionId": item["version_id"]} if item.get("version_id") else None
    s3.download_file(bucket, key, str(path), ExtraArgs=extra)
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(chunk)
    if digest.hexdigest() != item["sha256"] or path.stat().st_size != item["bytes"]:
        raise ValueError(f"El original descargado no coincide con su recibo: {uri}")
    return path


def load_districts(path):
    if path.suffix.lower() in {".rar", ".zip"}:
        names = subprocess.check_output(["bsdtar", "-tf", str(path)], text=True).splitlines()
        geopackages = [name for name in names if name.lower().endswith(".gpkg")]
        if len(geopackages) != 1:
            raise ValueError("El archivo distrital debe contener un solo GeoPackage.")
        output = path.with_suffix(".gpkg")
        with output.open("wb") as stream:
            subprocess.run(["bsdtar", "-xOf", str(path), geopackages[0]], stdout=stream, check=True)
        path = output
    data = gpd.read_file(path)
    data.columns = data.columns.str.strip().str.lower()
    data = data[["ubigeo", "nombdep", "nombprov", "nombdist", "geometry"]].to_crs(4326)
    data["ubigeo"] = data.ubigeo.astype("string").str.strip().str.zfill(6)
    if not data.ubigeo.str.fullmatch(r"\d{6}").fillna(False).all() or data.ubigeo.duplicated().any():
        raise ValueError("La cartografía debe contener UBIGEO distritales únicos de seis dígitos.")
    if data.geometry.isna().any() or data.geometry.is_empty.any() or data.empty:
        raise ValueError("La cartografía contiene geometrías ausentes.")
    repair = ~data.geometry.is_valid
    data.loc[repair, "geometry"] = data.loc[repair].geometry.buffer(0)
    if (not data.geometry.is_valid.all() or data.geometry.is_empty.any()
            or not data.geometry.geom_type.isin(["Polygon", "MultiPolygon"]).all()):
        raise ValueError("La cartografía no pudo normalizarse a polígonos válidos.")
    data["superficie_km2"] = data.to_crs(6933).area / 1_000_000
    return data.sort_values("ubigeo").reset_index(drop=True)


def district_frame(spark, data):
    rows = [(r.ubigeo, r.nombdep, r.nombprov, r.nombdist, r.superficie_km2) for r in data.itertuples()]
    return spark.createDataFrame(rows, "ubigeo string, departamento string, provincia string, "
                                       "distrito string, superficie_km2 double")


def assign_districts(frame, geo):
    schema = StructType(list(frame.schema.fields)).add("ubigeo", "string").add("coincidencias", "integer")

    def locate(batches):
        codes, polygons = geo.value
        tree = shapely.STRtree(shapely.from_wkb(polygons))
        for batch in batches:
            points = shapely.points(batch.tile_x.to_numpy(), batch.tile_y.to_numpy())
            point_index, polygon_index = tree.query(points, predicate="covered_by")
            matches = np.bincount(point_index, minlength=len(batch)).astype("int32")
            unique = matches[point_index] == 1
            district = np.full(len(batch), None, dtype=object)
            district[point_index[unique]] = codes[polygon_index[unique]]
            yield batch.assign(ubigeo=district, coincidencias=matches)

    return frame.mapInPandas(locate, schema)


def population_frame(spark, districts, path):
    rows = []
    with rasterio.open(path) as dataset:
        if dataset.count != 1 or dataset.crs is None or dataset.nodata is None:
            raise ValueError("WorldPop requiere una banda con CRS y valor NoData.")
        for district in districts.to_crs(dataset.crs).itertuples():
            try:
                pixels, _ = mask(dataset, [district.geometry], crop=True, filled=False)
                values = pixels.compressed()
                values = values[np.isfinite(values) & (values >= 0)]
            except ValueError as error:
                if "do not overlap" not in str(error):
                    raise
                values = np.array([])
            estimate = float(np.sum(values, dtype=np.float64)) if values.size else None
            rows.append((district.ubigeo, estimate, int(values.size)))
    return spark.createDataFrame(rows, "ubigeo string, poblacion_2020 double, pixeles_validos long")

def read_mobile(spark, paths, bounds):
    raw = spark.read.parquet(*paths)
    pattern = r"/(\d{4})-(01|04|07|10)-01_performance_mobile_tiles\.parquet$"
    year = F.regexp_extract(F.input_file_name(), pattern, 1).cast("int")
    month = F.regexp_extract(F.input_file_name(), pattern, 2).cast("int")
    tiles = raw.select("quadkey", F.col("avg_d_kbps").cast("long").alias("avg_d_kbps"),
                       F.col("tests").cast("long").alias("tests"),
                       F.col("devices").cast("long").alias("devices"),
                       (year * 4 + ((month - 1) / 3).cast("int")).alias("period_id"))

    # La tesela de zoom 6 filtra el área amplia, incluso en archivos sin tile_x ni tile_y.
    west, south, east, north = map(float, bounds)
    x0, x1 = [int((lon + 180) * 64 / 360) for lon in (west, east)]
    y0, y1 = [int((1 - math.asinh(math.tan(math.radians(lat))) / math.pi) * 32) for lat in (north, south)]
    prefixes = ["".join(str(((x >> bit) & 1) + 2 * ((y >> bit) & 1)) for bit in range(5, -1, -1))
                for x in range(x0, x1 + 1) for y in range(y0, y1 + 1)]
    candidates = tiles.filter(F.substring("quadkey", 1, 6).isin(prefixes) & F.col("quadkey").rlike(r"^[0-3]{16}$"))
    x = F.expr("aggregate(sequence(1, 16), 0, (a, i) -> a * 2 + (int(substring(quadkey, i, 1)) & 1))")
    y = F.expr("aggregate(sequence(1, 16), 0, (a, i) -> a * 2 + int(int(substring(quadkey, i, 1)) DIV 2))")
    lat = F.degrees(F.atan(F.sinh(math.pi * (1 - 2 * (y + 0.5) / 65536))))
    return (candidates.withColumn("tile_x", (x + 0.5) / 65536 * 360 - 180)
            .withColumn("tile_y", lat)
            .filter(F.col("tile_x").between(west, east) & F.col("tile_y").between(south, north)))



def read_bronze_inputs(manifest, work, selected_period=None):
    mobile, coverage, district, worldpop = validate_inventory(manifest["objects"])
    if selected_period:
        mobile = [item for item in mobile if PurePosixPath(item["key"]).name.startswith(selected_period)]
        if len(mobile) != 1:
            raise ValueError(f"Bronze no contiene el trimestre solicitado: {selected_period}")
    paths = [verified_uri(item) for item in mobile]
    periods = sorted(object_period(item) for item in mobile)
    return paths, periods, verified_uri(coverage), load_districts(downloaded_object(district, work)), downloaded_object(worldpop, work)
print("Verificando objetos en S3 y preparando los insumos geográficos...", flush=True)
temporary = Path(resources.enter_context(TemporaryDirectory(prefix="silver-", dir=TEMP_DIR)))
paths, period_ids, coverage, geography, raster = read_bronze_inputs(bronze, temporary)
silver_root = f"s3://{SILVER_BUCKET}/run_id={RUN_ID}"
ops_root = f"s3://{OPERATIONS_BUCKET}/run_id={RUN_ID}/silver"
with rasterio.open(raster) as ds:
    show_values({"Archivos Ookla verificados": len(paths), "Distritos": len(geography),
                 "CRS de polígonos": str(geography.crs), "CRS de WorldPop": str(ds.crs),
                 "Raster (ancho × alto)": f"{ds.width} × {ds.height}", "Valor NoData": ds.nodata,
                 "Corte trimestral": f"{period_ids[0]//4}-T{period_ids[0]%4+1} a {period_ids[-1]//4}-T{period_ids[-1]%4+1}"})

Verificando objetos en S3 y preparando los insumos geográficos...


,Indicador,Valor
0,Archivos Ookla verificados,29
1,Distritos,1890
2,CRS de polígonos,EPSG:4326
3,CRS de WorldPop,EPSG:4326
4,Raster (ancho × alto),15214 × 21977
5,Valor NoData,-99999.0
6,Corte trimestral,2019-T1 a 2026-T1


## 5. Limpieza y asignación geográfica de Ookla

El universo de calidad empieza en los candidatos que pasan el filtro geográfico. Cada tesela válida se asigna mediante su centroide.

In [7]:
def clean_tiles(raw, geo, silver_root, ops_root):
    keys = ["period_id", "quadkey"]
    if "replica_id" in raw.columns:
        keys.append("replica_id")
    inspected = raw.withColumn(
        "motivo",
        F.when(F.col("period_id").isNull(), "periodo_invalido")
        .when(F.col("tests").isNull() | (F.col("tests") <= 0) | F.col("devices").isNull()
              | (F.col("devices") <= 0) | (F.col("devices") > F.col("tests")), "pruebas_o_dispositivos_invalidos")
        .when(F.col("avg_d_kbps").isNull() | (F.col("avg_d_kbps") <= 0), "descarga_invalida")
        .when(F.count("*").over(Window.partitionBy(*keys)) > 1, "clave_duplicada"),
    ).persist(StorageLevel.DISK_ONLY)
    assigned = assign_districts(inspected.filter("motivo IS NULL").drop("motivo"), geo).withColumn(
        "motivo",
        F.when(F.col("coincidencias") == 0, "fuera_de_poligonos")
        .when(F.col("coincidencias") > 1, "asignacion_ambigua"),
    ).persist(StorageLevel.DISK_ONLY)
    try:
        rules = {row.motivo or "validas": row["count"] for row in inspected.groupBy("motivo").count().collect()}
        spatial = {row.motivo or "asignadas": row["count"] for row in assigned.groupBy("motivo").count().collect()}
        if not spatial.get("asignadas") or sum(spatial.values()) != rules.get("validas"):
            raise ValueError("La asignación espacial no produjo teselas válidas o los conteos no concilian.")
        write_table(inspected.filter("motivo IS NOT NULL"), f"{ops_root}/cuarentena/calidad")
        write_table(assigned.filter("motivo IS NOT NULL"), f"{ops_root}/cuarentena/espacial")
        clean = assigned.filter("motivo IS NULL").drop("motivo", "coincidencias")
        uri = write_table(clean, f"{silver_root}/ookla_teselas", ["period_id"])
        completeness = inspected.agg(*[F.count(c).alias(c) for c in
                                    ("period_id", "quadkey", "avg_d_kbps", "tests", "devices")]).first().asDict()
        return uri, {"candidatas": sum(rules.values()), "reglas": rules, "espacial": spatial,
                     "completitud_candidatas": completeness}
    finally:
        assigned.unpersist()
        inspected.unpersist()


def clean_coverage(spark, csv_uri, silver_root):
    raw = (spark.read.option("header", "true").option("sep", ";").option("encoding", "windows-1252")
           .option("mode", "FAILFAST").csv(csv_uri))
    coverage = raw.select(
        F.when(F.trim("UBIGEO_DISTRITO").rlike(r"^\d{1,6}$"),
               F.lpad(F.trim("UBIGEO_DISTRITO"), 6, "0")).alias("ubigeo"),
        F.when(F.trim("UBIGEO_CCPP").rlike(r"^\d{1,10}$"),
               F.lpad(F.trim("UBIGEO_CCPP"), 10, "0")).alias("centro_poblado"),
        F.trim("EMPRESA_OPERADORA").alias("operador"),
        F.trim("PERIODO").alias("periodo_referencia"),
        F.expr("try_to_timestamp(FECHA_CORTE, 'yyyyMMdd')").cast("date").alias("fecha_corte_fuente"),
        F.when(F.trim("4G").isin("0", "1"), F.trim("4G").cast("int")).alias("declara_4g"),
    )
    uri = write_table(coverage, f"{silver_root}/osiptel_cobertura")
    stored = spark.read.parquet(uri)
    counts = stored.agg(
        F.count("*").alias("filas"), F.count("ubigeo").alias("con_ubigeo"),
        F.count("declara_4g").alias("con_dato_4g"),
    ).first().asDict()
    dates = stored.select("periodo_referencia", "fecha_corte_fuente").distinct().take(2)
    if len(dates) != 1 or not dates[0].periodo_referencia or not dates[0].fecha_corte_fuente:
        raise ValueError("OSIPTEL debe contener una sola instantánea con fecha de corte.")
    return uri, counts, dates[0].periodo_referencia, str(dates[0].fecha_corte_fuente)


def silver_tiles(spark, paths, geography, silver_root, ops_root, scale=1):
    geo = spark.sparkContext.broadcast((geography.ubigeo.to_numpy(), geography.geometry.to_wkb().tolist()))
    try:
        raw = read_mobile(spark, paths, geography.total_bounds)
        if scale > 1:
            raw = raw.withColumn("replica_id", F.explode(F.sequence(F.lit(1), F.lit(scale))))
        tiles_uri, quality = clean_tiles(raw, geo, silver_root, ops_root)
    finally:
        geo.destroy()
    return tiles_uri, quality

tiles_uri, quality = silver_tiles(spark, paths, geography, silver_root, ops_root)
show_values({"Candidatas en el filtro territorial": quality["candidatas"],
             "Asignadas a un distrito": quality["espacial"]["asignadas"],
             "Salida Parquet": tiles_uri, "Cuarentenas": f"{ops_root}/cuarentena/"})
show_rows([{"control": k, "filas": v} for k, v in quality["reglas"].items()])
show_rows([{"resultado geográfico": k, "filas": v} for k, v in quality["espacial"].items()])
show_rows([{"campo": k, "valores presentes": v, "completitud_pct": round(100 * v / quality["candidatas"], 2)}
           for k, v in quality["completitud_candidatas"].items()])
spark.read.parquet(tiles_uri).select("ubigeo", "period_id", "quadkey", "avg_d_kbps", "tests").show(8, truncate=False)

,Indicador,Valor
0,Candidatas en el filtro territorial,555697
1,Asignadas a un distrito,417420
2,Salida Parquet,s3://american-tower-dev-silver-2ad71fded155d2bd52df8f836e/run_id=20261125T072542Z-e92a810559/ookla_teselas
3,Cuarentenas,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T072542Z-e92a810559/silver/cuarentena/


,control,filas
0,validas,555697


,resultado geográfico,filas
0,asignadas,417420
1,fuera_de_poligonos,138277


,campo,valores presentes,completitud_pct
0,period_id,555697,100.0
1,quadkey,555697,100.0
2,avg_d_kbps,555697,100.0
3,tests,555697,100.0
4,devices,555697,100.0


+------+---------+----------------+----------+-----+
|ubigeo|period_id|quadkey         |avg_d_kbps|tests|
+------+---------+----------------+----------+-----+
|240302|8101     |2100013013323332|14998     |2    |
|200706|8101     |2100013023212331|6719      |1    |
|200706|8101     |2100013023213132|35268     |1    |
|200701|8101     |2100013203000310|23130     |4    |
|200506|8101     |2100013203323012|63772     |4    |
|200603|8101     |2100013203331023|37284     |13   |
|200603|8101     |2100013203331123|23395     |18   |
|200114|8101     |2100013213233110|42254     |1    |
+------+---------+----------------+----------+-----+
only showing top 8 rows



## 6. Dimensiones y cobertura OSIPTEL

In [8]:
def silver_context_tables(spark, period_ids, coverage, geography, raster, silver_root):
    districts = district_frame(spark, geography)
    periods = spark.createDataFrame([(period,) for period in period_ids], "period_id int")
    coverage_uri, coverage_quality, reference, cutoff = clean_coverage(spark, coverage, silver_root)
    tables = {
        "distritos": write_table(districts, f"{silver_root}/distritos"),
        "trimestres": write_table(periods, f"{silver_root}/trimestres"),
        "osiptel_cobertura": coverage_uri,
    }
    return tables, coverage_quality, reference, cutoff

tables, coverage_quality, reference, cutoff = silver_context_tables(spark, period_ids, coverage, geography, raster, silver_root)
show_values({**coverage_quality, "Período declarado": reference, "Fecha de corte": cutoff})
show_rows([{"tabla": name, "uri": uri} for name, uri in tables.items()])
spark.read.parquet(tables["osiptel_cobertura"]).select("ubigeo", "centro_poblado", "operador", "declara_4g").show(8, truncate=False)

,Indicador,Valor
0,filas,51366
1,con_ubigeo,51366
2,con_dato_4g,51366
3,Período declarado,202303
4,Fecha de corte,2023-06-13


,tabla,uri
0,distritos,s3://american-tower-dev-silver-2ad71fded155d2bd52df8f836e/run_id=20261125T072542Z-e92a810559/distritos
1,trimestres,s3://american-tower-dev-silver-2ad71fded155d2bd52df8f836e/run_id=20261125T072542Z-e92a810559/trimestres
2,osiptel_cobertura,s3://american-tower-dev-silver-2ad71fded155d2bd52df8f836e/run_id=20261125T072542Z-e92a810559/osiptel_cobertura


+------+--------------+-------------------+----------+
|ubigeo|centro_poblado|operador           |declara_4g|
+------+--------------+-------------------+----------+
|010101|0101010001    |Viettel Perú S.A.C.|1         |
|010101|0101010002    |Viettel Perú S.A.C.|0         |
|010101|0101010004    |Viettel Perú S.A.C.|1         |
|010101|0101010006    |Viettel Perú S.A.C.|1         |
|010101|0101010007    |Viettel Perú S.A.C.|0         |
|010101|0101010008    |Viettel Perú S.A.C.|0         |
|010101|0101010009    |Viettel Perú S.A.C.|0         |
|010101|0101010010    |Viettel Perú S.A.C.|0         |
+------+--------------+-------------------+----------+
only showing top 8 rows



## 7. Población distrital WorldPop 2020

In [9]:
def silver_population(spark, geography, raster, silver_root):
    uri = write_table(population_frame(spark, geography, raster), f"{silver_root}/worldpop_2020")
    stats = spark.read.parquet(uri).agg(
        F.count("*").alias("distritos"), F.count("poblacion_2020").alias("con_poblacion"),
        F.sum("pixeles_validos").alias("pixeles_validos"),
    ).first().asDict()
    if stats["distritos"] != len(geography) or not stats["con_poblacion"]:
        raise ValueError("WorldPop Silver no concilia el universo distrital o no contiene población.")
    return uri, stats

tables["worldpop_2020"], population_stats = silver_population(spark, geography, raster, silver_root)
show_values({**population_stats, "Referencia": 2020, "Salida": tables["worldpop_2020"]})

,Indicador,Valor
0,distritos,1890
1,con_poblacion,1890
2,pixeles_validos,151864618
3,Referencia,2020
4,Salida,s3://american-tower-dev-silver-2ad71fded155d2bd52df8f836e/run_id=20261125T072542Z-e92a810559/worldpop_2020


## 8. Conciliación, recursos y publicación Silver

In [10]:
stored = spark.read.parquet(tiles_uri)
persisted = stored.count()
if persisted != quality["espacial"]["asignadas"]:
    raise ValueError("La cantidad de teselas persistidas difiere del control de asignación.")
usage = executor_usage(spark.sparkContext)
tables["ookla_teselas"] = tiles_uri
manifest = {"tables": tables, "quality": quality, "coverage_quality": coverage_quality,
            "population_quality": population_stats, "osiptel_periodo": reference, "osiptel_fecha_corte": cutoff,
            "distritos": len(geography), "trimestres": len(period_ids), "schema": "silver-autonomo-v1",
            "complete": True, "run_id": RUN_ID, "bronze_run_id": bronze["run_id"],
            "bronze_manifest_uri": BRONZE_RUN_URI, "application_id": spark.sparkContext.applicationId,
            "spark_event_log_uri": events_uri, "completed_at_utc": datetime.now(timezone.utc).isoformat()}
resultado = {**manifest, "manifest_uri": save_json(f"{ops_root}/_SUCCESS.json", manifest)}
show_values({"Teselas verificadas en Parquet": persisted, "Tablas Silver": len(tables),
             "Executors activos al final": usage["activos"], "Estado del monitoreo": usage["estado"],
             "Manifiesto Silver": resultado["manifest_uri"]})
print(f"SILVER_RUN_URI = {resultado['manifest_uri']!r}")
resources.close()

,Indicador,Valor
0,Teselas verificadas en Parquet,417420
1,Tablas Silver,5
2,Executors activos al final,8
3,Estado del monitoreo,lectura disponible
4,Manifiesto Silver,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T072542Z-e92a810559/silver/_SUCCESS.json


SILVER_RUN_URI = 's3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T072542Z-e92a810559/silver/_SUCCESS.json'
